<a href="https://colab.research.google.com/github/Aniketh78/Generative-AI-Lab_Experiments/blob/main/genAiExp07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:

import os
import faiss
import time
import numpy as np

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)


os.environ["GOOGLE_API_KEY"] = "sodjadnuaejbu-nwjdb-thisaintmyapikeydms"


loader = PyPDFLoader("/content/Week1_Introduction_To_EBusiness.pdf")
documents = loader.load()


splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
texts = splitter.split_documents(documents)
chunks = [doc.page_content for doc in texts]


embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

print(f"Embedding {len(chunks)} chunks...")
all_embeddings = []
batch_size = 5

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    batch_embeddings = embedding_model.embed_documents(batch)
    all_embeddings.extend(batch_embeddings)
    print(f"Processed {min(i + batch_size, len(chunks))}/{len(chunks)} chunks...")
    time.sleep(2)

embeddings = np.array(all_embeddings).astype("float32")
faiss.normalize_L2(embeddings)


dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)


query = "What is E-Business?"


query_embedding = embedding_model.embed_query(query)
query_embedding = np.array([query_embedding]).astype("float32")
faiss.normalize_L2(query_embedding)

k = 3
scores, indices = index.search(query_embedding, k)
retrieved_chunks = [chunks[i] for i in indices[0]]


context = "\n\n".join(retrieved_chunks)


llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
prompt = f"""Answer the question based ONLY on the context below.\n\nContext:\n{context}\n\nQuestion:\n{query}"""
response = llm.invoke(prompt)

print("\nAnswer:\n", response.content)

print("\nRetrieved Context:\n")
for chunk in retrieved_chunks:
    print(chunk[:300])
    print("-" * 50)

Embedding 111 chunks...
Processed 5/111 chunks...
Processed 10/111 chunks...
Processed 15/111 chunks...
Processed 20/111 chunks...
Processed 25/111 chunks...
Processed 30/111 chunks...
Processed 35/111 chunks...
Processed 40/111 chunks...
Processed 45/111 chunks...
Processed 50/111 chunks...
Processed 55/111 chunks...
Processed 60/111 chunks...
Processed 65/111 chunks...
Processed 70/111 chunks...
Processed 75/111 chunks...
Processed 80/111 chunks...
Processed 85/111 chunks...
Processed 90/111 chunks...
Processed 95/111 chunks...
Processed 100/111 chunks...
Processed 105/111 chunks...
Processed 110/111 chunks...
Processed 111/111 chunks...

Answer:
 [{'type': 'text', 'text': 'Based on the provided context, E-Business (a term coined by Lou Gerstner, CEO of IBM) is defined as:\n\n*   **A superset of e-commerce:** It is more than just buying and selling; it includes any business process that relies on an automated information system (mostly Web-based technologies).\n*   **A value chain sp